In [ ]:
import sys

sys.path.append("..")  # to import from parent dir
import numpy as np
from IPython.display import HTML
import tonic
import matplotlib.pyplot as plt
import os.path as osp
from scipy.ndimage import gaussian_filter, sobel

from triangle import TriangleMovement
from utils.visualize_utils import animate_events
from utils.data_utils import *
from idn.loader.loader_dsec import HarrisRecursive

In [ ]:
# Image/grid size
img_height = 200
img_width = 200
image_size = (img_height, img_width)

center = np.array([img_width // 2, img_height // 2])

num_frames = 2_000

alpha = 35 * np.pi / 180
triangle_base = 400
triangle_height = triangle_base / 2 / np.tan(alpha / 2)

# Trajectory of the local shape
added_height = 200
start_pos = np.array([center[0], 0])
movement_direction = 10

In [ ]:
triangle = TriangleMovement(
    total_frames=num_frames,
    image_size=(img_height, img_width),
    triangle_base=triangle_base,
    triangle_height=triangle_height,
    added_height=added_height,
    start_pos=start_pos,
    face_color="black",
    speed=1.0,
    movement_direction=movement_direction,
)

# Display animation

anim = triangle.create_animation(frame_step=20)
HTML(anim.to_jshtml())

In [ ]:
data_array = triangle.generate_events()

transform = tonic.transforms.ToFrame(
    sensor_size=(triangle.image_size[1], triangle.image_size[0], 2),
    time_window=10,
    # event_count=500,
    overlap=0.5,
)
anim = animate_events(data_array, transform, fig_size=(5,5), invert_yaxis=True)
anim.save(filename="event_triangle.gif", writer="pillow")
HTML(anim.to_jshtml())

In [ ]:
sample = data_array[(data_array['t'] >= 500) & (data_array['t'] <= 503)]
plt.quiver(sample['x'], sample['y'], 50* sample['v_x'], 50*sample['v_y'], 
           color=np.where(sample['p'], 'r', 'b'), angles='xy', scale_units='xy', scale=1)
# plt.gca().invert_yaxis()
plt.axis('equal')
plt.title("Optical Flow of Events at t = 500-503 mS")
plt.savefig('optical_flow_at_200.png', dpi=300)
# plt.xlim(0, img_width)
# plt.ylim(0, img_height)
plt.show()

In [ ]:
data_array = triangle.generate_events()

In [ ]:
filter_size = 7
tau = 30

In [ ]:
harris_rec = HarrisRecursive(tau, filter_size, image_size)
harris_rec(data_array)
harris_eig1 = harris_rec.eig1
harris_eig2 = harris_rec.eig2
filter_values = harris_rec.filter_value_recursive

In [ ]:
idx = (data.pos[..., -1] > 500) & (data.pos[..., -1] < 510)
scatter = plt.scatter(data.pos[idx,0], data.pos[idx,1], c=harris_eig1[idx], cmap='viridis', marker='.', s=5)  # `c` maps color, `s` controls size
plt.colorbar(scatter, label='Value')
plt.xlabel('x [pixels]')
plt.ylabel('y [pixels]')
plt.title('eig1')
plt.axis('equal')
# plt.savefig(osp.join(saving_folder, f'filter_values.png'), dpi=300)
# Displaying the plot
plt.show()

scatter = plt.scatter(data.pos[idx,0], data.pos[idx,1], c=harris_eig2[idx], cmap='viridis', marker='.', s=5)  # `c` maps color, `s` controls size
plt.colorbar(scatter, label='Value')
plt.xlabel('x [pixels]')
plt.ylabel('y [pixels]')
plt.title('eig2')
plt.axis('equal')
# plt.savefig(osp.join(saving_folder, f'filter_values.png'), dpi=300)
# Displaying the plot
plt.show()

scatter = plt.scatter(data.pos[idx,0], data.pos[idx,1], c=filter_values[idx], cmap='viridis', marker='.', s=5)  # `c` maps color, `s` controls size
plt.colorbar(scatter, label='Value')
plt.xlabel('x [pixels]')
plt.ylabel('y [pixels]')
plt.title('filter values')
plt.axis('equal')
# plt.savefig(osp.join(saving_folder, f'filter_values.png'), dpi=300)
# Displaying the plot
plt.show()

v_mag = torch.norm(data['v'], dim=1)
scatter = plt.scatter(data.pos[idx,0], data.pos[idx,1], c=v_mag[idx], cmap='viridis', marker='.', s=5)  # `c` maps color, `s` controls size
plt.colorbar(scatter, label='Value')
plt.xlabel('x [pixels]')
plt.ylabel('y [pixels]')
plt.title('Magnitude of Velocity')
plt.axis('equal')
# plt.savefig(osp.join(saving_folder, f'magnitude_velocity.png'), dpi=300)
# Displaying the plot
plt.show()

sample = data_array[idx]
plt.quiver(sample['x'], sample['y'], 10* sample['v_x'], 10*sample['v_y'], 
           color=np.where(sample['p'], 'r', 'b'), angles='xy', scale_units='xy', scale=1)
plt.axis('equal')
plt.title("Optical Flow of Events")
# plt.savefig(osp.join(saving_folder, f'optical_flow_directions.png'), dpi=300)
plt.show()

In [ ]:
frames = np.arange(0, 750, 40)
all_points = np.stack([triangle.trajectory_at(f)[0] for f in frames], axis=0)

flows = np.concatenate([triangle.compute_optical_flow(np.expand_dims(p,axis=0),f) for p,f in zip(all_points, frames)], axis=0)

# Plot
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(0, triangle.image_size[1])
ax.set_ylim(0, triangle.image_size[0])
ax.set_aspect('equal')
ax.axis('off')


# Draw optical flow vectors
ax.quiver(all_points[:, 0], all_points[:, 1],
          40*flows[:, 0], 40*flows[:, 1],
          angles='xy', scale_units='xy', scale=1, color='blue')

plt.show()

In [ ]:
def spatiotemporal_image_at(data_array, portion = 0.25):
    
    t_max = data_array['t'][-1].item()
    t_min = data_array['t'][0].item()

    print(f'max. time: {t_max}')
    t_obs = (t_max - t_min) * portion + t_min
    print(f'observed time: {t_obs}')
    idx = data_array['t'] < t_obs
    data_truncated = data_array[idx].copy()
    print(f'num of events: {data_truncated.shape[0]}')
    
    harris_rec(data_truncated)
    # Compute the temporal lag
    temporal_lag = np.exp(- (t_obs - harris_rec.last_time_tensor)/harris_rec.tau)

    # update the temporal accumulation tensor

    spatiotemporal_image = harris_rec.temporal_accumulation_tensor * temporal_lag
    # crop the image to the center image_size = (img_height, img_width)
    center_y = spatiotemporal_image.shape[1] // 2
    center_x = spatiotemporal_image.shape[2] // 2
    spatiotemporal_image = spatiotemporal_image[:,center_y - img_height // 2:center_y + img_height // 2,
                          center_x - img_width // 2:center_x + img_width // 2]

    return spatiotemporal_image, data_truncated

In [ ]:
import matplotlib.gridspec as gridspec
speed = 10.0
portion = 0.25
triangle.speed = speed
data_array = triangle.change_event_speeds()
spatiotemporal_image, data_truncated = spatiotemporal_image_at(data_array, portion=portion)
print(f'spatiotemporal_image shape: {spatiotemporal_image.shape}')
frame = int(triangle.total_frames * portion)
print(f'frame: {frame}')
points = triangle.trajectory_at(frame)[0].reshape(1,2)
print(f'points: {points}')
flows = triangle.compute_optical_flow(points,frame)
flows_norm = flows / np.linalg.norm(flows, axis=1, keepdims=True)
print(f'flows: {flows}')

# Determine common vmin and vmax
vmin = min(data.min() for data in spatiotemporal_image)
vmax = max(data.max() for data in spatiotemporal_image)


# Set up figure with GridSpec
fig = plt.figure(figsize=(15, 8))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05])

v_left = np.array([-triangle_height,triangle_base / 2])
v_left /= np.linalg.norm(v_left)

v_right = np.array([triangle_height,triangle_base / 2])
v_right /= np.linalg.norm(v_right)

axes = []
for i in range(2):
    ax = fig.add_subplot(gs[i])
    im = ax.imshow(spatiotemporal_image[i], vmin=vmin, vmax=vmax, cmap='viridis')
    ax.set_title(f"polarity {i}")
    ax.invert_yaxis()
    # ax.axis('off')
    axes.append(ax)
    # Draw optical flow vectors
    ax.quiver(points[:, 0], points[:, 1],
        40*flows_norm[:, 0], 40*flows_norm[:, 1],
        angles='xy', scale_units='xy', scale=1, color='red')
    ax.quiver(points[:, 0], points[:, 1],
        40*v_left[0], 40*v_left[1],
        angles='xy', scale_units='xy', scale=1, color='blue')
    ax.quiver(points[:, 0], points[:, 1],
        40*v_right[0], 40*v_right[1],
        angles='xy', scale_units='xy', scale=1, color='blue')

# Add colorbar in the third slot
cax = fig.add_subplot(gs[2])
fig.colorbar(im, cax=cax)

fig.suptitle(f'Spatiotemporal image @ speed = {speed} frame/mS', fontsize=16)

# Adjust layout to accommodate title
plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space for suptitle

# fig.savefig(osp.join(saving_folder, f'spatiotemporal_image_speed_{speed}_portion_{portion}.png'))

plt.show()


In [ ]:
V = np.stack([v_left,v_right], axis=1)
V_inv_t = np.linalg.pinv(V).T

In [ ]:
def transform_structures(Sxx, Sxy, Syy, V):
    """
    Compute C = V A V^T for each pixel where A = [[Sxx, Sxy],[Sxy, Syy]].
    Inputs:
      Sxx, Sxy, Syy : np.ndarray of shape (H, W)
      V             : np.ndarray of shape (2,2)
    Returns:
      Cxx, Cxy, Cyy : np.ndarray of shape (H, W)
    """
    # validate shapes
    if Sxx.shape != Sxy.shape or Sxx.shape != Syy.shape:
        raise ValueError("Sxx, Sxy, Syy must have the same shape (H,W).")
    if V.shape != (2,2):
        raise ValueError("V must be shape (2,2).")
    
    a, b = V[0,0], V[0,1]
    c, d = V[1,0], V[1,1]

    # compute outputs (vectorized)
    Cxx = (a*a) * Sxx + 2*a*b * Sxy + (b*b) * Syy
    Cxy = (a*c) * Sxx + (a*d + b*c) * Sxy + (b*d) * Syy
    Cyy = (c*c) * Sxx + 2*c*d * Sxy + (d*d) * Syy

    return Cxx, Cxy, Cyy


In [ ]:
def find_values(sum_val, prod_val):
    """
    Given the sum and product of two numbers,
    return the two numbers in ascending order.
    """
    # Discriminant check
    discriminant = sum_val**2 - 4 * prod_val
    if np.any(discriminant < 0):
        raise ValueError("No real solutions exist for the given sum and product.")
    
    # Quadratic formula
    x1 = (sum_val + np.sqrt(discriminant)) / 2
    x2 = (sum_val - np.sqrt(discriminant)) / 2
    
    return np.minimum(x1, x2), np.maximum(x1, x2)


In [ ]:
def plot_structures(Cxx, Cxy, Cyy):
    fig, axes = plt.subplots(3, 1, figsize=(6,20))

    images = [Cxx, Cxy, Cyy]
    titles = ['Cxx', 'Cxy', 'Cyy']

    for ax, img, title in zip(axes, images, titles):
        im = ax.imshow(img, origin='upper', cmap='viridis')  # 'origin=upper' inverts y-axis
        ax.set_title(title)
        ax.axis('off')
        ax.invert_yaxis()
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)  # add colorbar per image

    plt.tight_layout()
    plt.show()


In [ ]:
plt.imshow(spatiotemporal_image[0], cmap='viridis')
plt.title('Spatiotemporal Image Polarity 0')
plt.axis('equal')
plt.colorbar(label='Value')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
image = spatiotemporal_image[0]

# Step 1: Compute image gradients
Ix = sobel(image, axis=1) # horizontal gradient
Iy = sobel(image, axis=0) # vertical gradient

# Step 2: Compute products of derivatives
Ixx = Ix**2
Iyy = Iy**2
Ixy = Ix*Iy

# Step 3: Apply Gaussian filter to smooth the products
Sxx = gaussian_filter(Ixx, sigma=1)
Syy = gaussian_filter(Iyy, sigma=1)
Sxy = gaussian_filter(Ixy, sigma=1)

plot_structures(Sxx, Sxy, Syy)

In [ ]:
Cxx_xrop = Cxy[int(points[0,1])-7:int(points[0,1])+8, int(points[0,0])-7:int(points[0,0])+8]
plt.imshow(Cxx_xrop, cmap='viridis')
plt.title('Cxx around point')
plt.axis('equal')
plt.colorbar(label='Value')
plt.gca().invert_yaxis()
plt.show()



In [ ]:
theta1 = np.random.uniform(0, np.pi/3)
v1 = np.array([np.cos(theta1), np.sin(theta1)])
theta2 = np.pi - theta1 #np.random.uniform(0, 2*np.pi)
v2 = np.array([np.cos(theta2), np.sin(theta2)])

V = np.stack([v1,v2], axis=1)
V_inv = np.linalg.pinv(V)
A = V @ np.array([[3, 0],[0, 7]]) @ V.T
print(A)
print(V_inv @ A @ V_inv.T)

print(np.tan(theta1)**2 * A[0,0] - A[1,1])

In [ ]:
tan_theta = v_right[1]/v_right[0]
plt.imshow(tan_theta**2 * Sxx - Syy, cmap='viridis')
plt.title('tan(theta)^2 * Sxx - Syy')
plt.axis('equal')
plt.colorbar(label='Value')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
theta1 = np.random.uniform(0, np.pi/2)
theta1 = np.arctan(v_right[1]/v_right[0])
v1 = np.array([np.cos(theta1), np.sin(theta1)])
theta2 = np.pi - theta1
v2 = np.array([np.cos(theta2), np.sin(theta2)])

V_inv_t_theory = 1/np.sin(theta2 - theta1)*np.array([[np.sin(theta2),-np.sin(theta1)],[-np.cos(theta2),np.cos(theta1)]]) 
V = np.stack([v1,v2], axis=1)


tan_theta = v_right[1]/v_right[0]
# V = np.stack([v_left,v_right], axis=1)
V_inv = np.linalg.pinv(V)


Cxx, Cxy, Cyy = transform_structures(Sxx=Sxx, Sxy=Sxy, Syy=Syy, V=V_inv)

plot_structures(Cxx, Cxy, Cyy)



In [ ]:
# Step 4: Compute Harris response
detM = (Sxx * Syy) - (Sxy**2)
traceM = Sxx + Syy

Eigs = find_values(traceM, detM)
prod_eig = Eigs[0] * Eigs[1] * np.sin(alpha)**2
sum_eig = Eigs[0] + Eigs[1]

coeffs = find_values(sum_val=sum_eig, prod_val=prod_eig)
v = np.array([triangle_height,triangle_base / 2])
v /= np.linalg.norm(v)
v_x = v[0] * (coeffs[1] - coeffs[0])
v_y = v[1] * (coeffs[0] + coeffs[1])

v = np.stack([v_x, v_y], axis=2)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def flow_to_color(flow, max_flow=None):
    """
    Convert flow to RGB image.
    flow: [H,W,2] numpy array
    """
    h, w = flow.shape[:2]
    fx, fy = flow[:,:,0], flow[:,:,1]

    mag, ang = cv2.cartToPolar(fx, fy, angleInDegrees=True)

    if max_flow is None:
        max_flow = np.max(mag)
        print(f"max flow: {max_flow}")

    hsv = np.zeros((h, w, 3), dtype=np.uint8)
    hsv[...,0] = ang / 2                  # Hue (0-180 in OpenCV)
    hsv[...,1] = 255                      # Saturation
    hsv[...,2] = np.clip((mag / max_flow) * 255, 0, 255)  # Value (brightness)

    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return rgb

def make_color_wheel(size=200):
    """
    Create a flow color wheel legend (direction -> hue, magnitude -> radius).
    """
    # coordinate grid
    x = np.linspace(-1, 1, size)
    y = np.linspace(-1, 1, size)
    xx, yy = np.meshgrid(y,-x)  # flip y for display

    flow = np.stack((xx, yy), axis=-1)  # [H,W,2]
    return flow_to_color(flow)

# --- Example usage ---
# Suppose you already have a flow field (H,W,2)
# For demo, we’ll make a simple synthetic flow:
H, W = 100, 150
flow = np.zeros((H,W,2), dtype=np.float32)
flow[...,0] = np.linspace(-5,5,W)   # horizontal ramp
flow[...,1] = np.linspace(-3,3,H)[:,None]  # vertical ramp

v_true = ((v * 0) + flows_norm[np.newaxis,...]) * (np.abs(v)>1)


flow_img = flow_to_color(v*100)
wheel_img = make_color_wheel(200)

# Plot
fig, ax = plt.subplots(1,2, figsize=(10,5))
ax[0].imshow(flow_img)
ax[0].set_title("Optical Flow Visualization")
# ax[0].axis("off")
ax[0].invert_yaxis()

ax[1].imshow(wheel_img)
ax[1].set_title("Color Wheel Legend")
ax[1].axis("off")
# ax[1].invert_yaxis()
plt.show()


In [ ]:
v

In [ ]:
# Visualization with color map

plt.imshow(v_y, cmap='viridis')
plt.colorbar(label='v_x values')
plt.title('v_x values across the image')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

img = spatiotemporal_image[0,:,:] * 256

# Visualization
plt.imshow(spatiotemporal_image[0,:,:], cmap='gray')
y, x = zip(*corners)
plt.scatter(x, y, c='red', s=10)
plt.title("Harris Corners")
plt.show()
